# Regression - part b

We start by loading the cleaned dataset from the first project.

In [79]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
import torch
from sklearn import model_selection
from sklearn.linear_model import Ridge
import scipy.stats as st


In [80]:
# Load dataset
df = pd.read_csv('/Volumes/MASTER/Year1/ML/ml-project/student_lifestyle_dataset..csv')

# Perform cleaning steps from project 1
df = df.drop(['Student_ID'], axis=1)
df = df[df['Physical_Activity_Hours_Per_Day'] < 11]

# Drop the discrete values for regression
df = df.drop(['Stress_Level', 'Gender'], axis=1)
df.head()

,Study_Hours_Per_Day,Extracurricular_Hours_Per_Day,Sleep_Hours_Per_Day,Social_Hours_Per_Day,Physical_Activity_Hours_Per_Day,Grades
0,6.9,3.8,8.7,2.8,1.8,7.48
1,5.3,3.5,8.0,4.2,3.0,6.88
2,5.1,3.9,9.2,1.2,4.6,6.68
3,6.5,2.1,7.2,1.7,6.5,7.20
4,8.1,0.6,6.5,2.2,6.6,8.78


For regression, we will analyze how well the study, extracurricular, sleep, social and physical activity hours per day predict their grades on a range between 5.0 and 10.0.

We define X as the matrix containing the continuous values Study_Hours_Per_Day, Extracurricular_Hours_Per_Day, Sleep_Hours_Per_Day, Social_Hours_Per_Day and Physical_Activity_Hours_Per_Day.

In [81]:
X = df.drop(['Grades'], axis=1).values

We define y as the Grades column.

In [82]:
y = df['Grades'].values

In [83]:
N, M = X.shape

*Implement two-level cross-validation (see algorithm 6 of the lecture notes). We will use
2-level cross-validation to compare the models with K1 = K2 = 10 folds2. As a baseline
model, we will apply a linear regression model with no features, i.e. it computes the mean
of y on the training data, and use this value to predict y on the test data*

*Make sure you can fit an ANN model to the data. As complexity-controlling parameter
for the ANN, we will use the number of hidden units h. Based on a few test-runs, select
a reasonable range of values for h (which should include h = 1), and describe the range of
values you will use for h and λ.*

Using the exercise 8, I modified the setup_storage_for_experiment to include the ANN.

In [84]:
def setup_storage_for_experiment(K_outer, K_inner, num_lambdas, num_hidden_units):
    # Setup storage for the optimal hyperparameters found from the inner CV
    optimal_lambdas = np.empty(K_outer)
    optimal_hidden_units = np.empty(K_outer)

    # Setup storage for model coefficients and errors for each experiment in all inner folds
    ws_inner = np.empty((M + 1, K_outer, K_inner, num_lambdas))
    ridge_train_errors_inner = np.empty((K_outer, K_inner, num_lambdas))
    ridge_test_errors_inner = np.empty((K_outer, K_inner, num_lambdas))
    ann_train_errors_inner = np.empty((K_outer, K_inner, num_hidden_units))
    ann_test_errors_inner = np.empty((K_outer, K_inner, num_hidden_units))

    # Setup storage for model coefficients for each experiment in all outer folds
    ws_outer = {
        'regularized': np.empty((M + 1, K_outer))
    }
    # Setup storage for errors as a dictionary
    errors_outer = {
        'train': {
            'baseline': np.empty((K_outer, 1)), 
            'regularized': np.empty((K_outer, 1)),
            'ann': np.empty((K_outer, 1))
        },
        'test': {
            'baseline': np.empty((K_outer, 1)), 
            'regularized': np.empty((K_outer, 1)),
            'ann': np.empty((K_outer, 1))
        }
    }
    return optimal_lambdas, optimal_hidden_units, ws_inner, ridge_train_errors_inner, ridge_test_errors_inner, ann_train_errors_inner, ann_test_errors_inner, ws_outer, errors_outer

I defined the ANN model to work with.

In [85]:
def get_model(input_dim, hidden_unit, output_dim):
    return torch.nn.Sequential(
        torch.nn.Linear(in_features=input_dim, out_features=hidden_unit, bias=True),     # Input layer
        torch.nn.Tanh(),                                                                 # Activation function
        torch.nn.Linear(in_features=hidden_unit, out_features=output_dim, bias=True),    # Output layer
    )

Using solutions from exercise 8 and 9, I have added inside the inner loop a code snippet to train ANN models with different hidden units. We have used standarization as suggested in the project 1 feedback.

In [ ]:
# Set random seed
np.random.seed(42)
torch.manual_seed(42)

# Values of regularization parameter lambda to test in the inner loop
lambdas = np.logspace(-2, 2, 40)

# Hidden units for ANN
hidden_units = [7, 8, 9, 10, 11]
lr = 1e-3
n_epochs = 1000

# Setup storage for the experiment
K_outer = 10
K_inner = 10
optimal_lambdas, optimal_hidden_units, ws_inner, ridge_train_errors_inner, ridge_test_errors_inner, ann_train_errors_inner, ann_test_errors_inner, ws_outer, errors_outer = setup_storage_for_experiment(K_outer, K_inner, len(lambdas), len(hidden_units))

# Create 10 fold splits
CV_outer = KFold(K_outer, shuffle=True, random_state=42)
CV_inner = KFold(K_inner, shuffle=True, random_state=42)

# Run two-layer cross-validation
for outer_fold_idx, (outer_train_index, outer_test_index) in enumerate(CV_outer.split(X, y)):
    
    # Extract training and test set for the current outer CV fold
    X_train_outer, y_train_outer = X[outer_train_index], y[outer_train_index]
    X_test_outer, y_test_outer = X[outer_test_index], y[outer_test_index]

    for inner_fold_idx, (inner_train_index, inner_test_index) in enumerate(CV_inner.split(X_train_outer, y_train_outer)):

        # Extract training and validation set for current inner CV fold
        X_train_inner, y_train_inner = X_train_outer[inner_train_index], y_train_outer[inner_train_index]
        X_test_inner, y_test_inner = X_train_outer[inner_test_index], y_train_outer[inner_test_index]

        # Compute the mean and standard deviation of the inner training data split, then standardize training and test sets
        mu_inner = np.mean(X_train_inner, axis=0)
        sigma_inner = np.std(X_train_inner, axis=0)
        
        # Standardize the inner training set and validation set
        X_train_inner_std = (X_train_inner - mu_inner) / sigma_inner
        X_test_inner_std = (X_test_inner - mu_inner) / sigma_inner

        # RIDGE REGRESSION
        # Loop over all values of lambda
        for lambda_idx, regularization_strength in enumerate(lambdas):
            # Create and fit the model
            lr_model = Ridge(alpha=regularization_strength)
            lr_model.fit(X_train_inner_std, y_train_inner)

            # Store the model coefficients for each value of lambda in the inner folds
            ws_inner[:, outer_fold_idx, inner_fold_idx, lambda_idx] = [lr_model.intercept_] + lr_model.coef_.flatten().tolist()

            # Compute and store the training and validation error
            ridge_train_errors_inner[outer_fold_idx, inner_fold_idx, lambda_idx] = np.mean((y_train_inner - lr_model.predict(X_train_inner_std))**2, axis=0)
            ridge_test_errors_inner[outer_fold_idx, inner_fold_idx, lambda_idx] = np.mean((y_test_inner - lr_model.predict(X_test_inner_std))**2, axis=0)
    
        # ANN
        X_train_inner_tensor = torch.tensor(X_train_inner_std, dtype=torch.float32)
        y_train_inner_tensor = torch.tensor(y_train_inner, dtype=torch.float32).view(-1, 1)
        X_test_inner_tensor = torch.tensor(X_test_inner_std, dtype=torch.float32)
        y_test_inner_tensor = torch.tensor(y_test_inner, dtype=torch.float32).view(-1, 1)

        for hidden_unit_idx, hidden_unit in enumerate(hidden_units):
            # Define a model with a specific number of hidden units
            ann_model = get_model(input_dim=M, hidden_unit=hidden_unit, output_dim=1)
            criterion = torch.nn.MSELoss()
            optimizer = torch.optim.SGD(params=ann_model.parameters(), lr=lr)
            
            for epoch in range(n_epochs):
                ann_model.train()
                outputs = ann_model(X_train_inner_tensor)
                loss = criterion(outputs, y_train_inner_tensor)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # Compute the final train and validation loss
            with torch.no_grad():
                ann_model.eval()
                train_outputs = ann_model(X_train_inner_tensor)
                val_outputs = ann_model(X_test_inner_tensor)
                train_loss = criterion(train_outputs, y_train_inner_tensor)
                val_loss = criterion(val_outputs, y_test_inner_tensor)
                
                # Store in the arrays
                ann_train_errors_inner[outer_fold_idx, inner_fold_idx, hidden_unit_idx] = train_loss.item()
                ann_test_errors_inner[outer_fold_idx, inner_fold_idx, hidden_unit_idx] = val_loss.item()


    # Determine the optimal value of lambda that gives the lowest test error on average from the inner folds
    optimal_lambda_idx = np.argmin(np.mean(ridge_test_errors_inner[outer_fold_idx], axis=0))
    optimal_lambda = lambdas[optimal_lambda_idx]
    
    # Store the optimal regularization strength for the current outer fold
    optimal_lambdas[outer_fold_idx] = optimal_lambda

    # Determine the optimal number of hidden units
    avg_val_errors = np.mean(ann_test_errors_inner[outer_fold_idx], axis=0)
    optimal_hidden_unit_idx = np.argmin(avg_val_errors)
    optimal_hidden_unit = hidden_units[optimal_hidden_unit_idx]
    optimal_hidden_units[outer_fold_idx] = optimal_hidden_unit

    # Compute the mean and standard deviation of the outer training data split
    mu_outer = np.mean(X_train_outer, axis=0)
    sigma_outer = np.std(X_train_outer, axis=0)

    # Standardize the outer training set and test set
    X_train_outer_std = (X_train_outer - mu_outer) / sigma_outer
    X_test_outer_std = (X_test_outer - mu_outer) / sigma_outer

    # 1. REGULARIZED LINEAR REGRESSION
    optimal_lr_model = Ridge(alpha=optimal_lambda)
    optimal_lr_model.fit(X_train_outer_std, y_train_outer)
    ws_outer['regularized'][:, outer_fold_idx] = [optimal_lr_model.intercept_] + optimal_lr_model.coef_.flatten().tolist()
    errors_outer['train']['regularized'][outer_fold_idx] = np.mean((y_train_outer - optimal_lr_model.predict(X_train_outer_std))**2, axis=0)
    errors_outer['test']['regularized'][outer_fold_idx] = np.mean((y_test_outer - optimal_lr_model.predict(X_test_outer_std))**2, axis=0)

    # 2. BASELINE (NO FEATURES)
    # Compute mean of training data
    y_train_mean = np.mean(y_train_outer)
    
    # Predict mean for all samples
    y_train_pred_baseline = np.full(len(y_train_outer), y_train_mean)
    y_test_pred_baseline = np.full(len(y_test_outer), y_train_mean)
    
    # Compute errors
    errors_outer['train']['baseline'][outer_fold_idx] = np.mean((y_train_outer - y_train_pred_baseline)**2)
    errors_outer['test']['baseline'][outer_fold_idx] = np.mean((y_test_outer - y_test_pred_baseline)**2)

    # 3. ANN
    # Convert to torch tensors
    X_train_outer_tensor = torch.tensor(X_train_outer_std, dtype=torch.float32)
    y_train_outer_tensor = torch.tensor(y_train_outer, dtype=torch.float32).view(-1, 1)
    X_test_outer_tensor = torch.tensor(X_test_outer_std, dtype=torch.float32)
    y_test_outer_tensor = torch.tensor(y_test_outer, dtype=torch.float32).view(-1, 1)

    optimal_ann_model = get_model(input_dim=M, hidden_unit=int(optimal_hidden_unit), output_dim=1)
    criterion = torch.nn.MSELoss()
    optimizer = torch.optim.SGD(params=optimal_ann_model.parameters(), lr=lr)
    
    for epoch in range(n_epochs):
        optimal_ann_model.train()
        outputs = optimal_ann_model(X_train_outer_tensor)
        loss = criterion(outputs, y_train_outer_tensor)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Evaluate ANN
    with torch.no_grad():
        optimal_ann_model.eval()
        train_outputs = optimal_ann_model(X_train_outer_tensor)
        test_outputs = optimal_ann_model(X_test_outer_tensor)
        errors_outer['train']['ann'][outer_fold_idx] = criterion(train_outputs, y_train_outer_tensor).item()
        errors_outer['test']['ann'][outer_fold_idx] = criterion(test_outputs, y_test_outer_tensor).item()

# Print results
print(f"\nOptimal regularization strengths per fold: {optimal_lambdas}")
print(f"Optimal hidden units per fold: {optimal_hidden_units}")


Optimal regularization strengths per fold: [3.34482759 3.75862069 4.5862069  4.10344828 4.10344828 3.
 5.         4.72413793 3.         5.        ]
Optimal hidden units per fold: [10.  7.  8.  8. 11.  7. 10.  8.  8. 10.]


In the testing phase we tried various intervals for lambdas and hidden units.

Hidden units: 
* [1, 2, 5, 10] > [10. 10. 10. 10. 10. 10. 10. 10. 10. 10.]
* [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20] > [ 9.  9.  8. 10.  9.  7. 11.  8. 10. 10.]
* [7, 8, 9, 10, 11] > [10.  7.  8.  8. 11.  7. 10.  8.  8. 10.]
* [7, 8, 10] > [10.  7. 10.  8. 10.  8.  7.  7.  7.  8.]
* [7, 8] > [8. 8. 7. 7. 8. 8. 7. 8. 7. 8.]

Lambdas:
* 10 ** -5 > 10 ** 8 > [ 1.  1.  1.  1.  1.  1. 10.  1.  1. 10.]
* 10 ** -1 -> 10 ** 2 > [4.12462638 4.12462638 4.12462638 4.12462638 4.12462638 2.42446202
 7.01703829 4.12462638 2.42446202 4.12462638]
* 1 -> 10 > [3.45510729 3.45510729 4.92388263 4.12462638 4.12462638 2.89426612
 7.01703829 4.92388263 2.89426612 5.87801607]
* 2 -> 6 > [3.37931034 3.79310345 4.62068966 4.06896552 4.06896552 2.96551724 6. 4.75862069 2.68965517 5.5862069 ]
* 3 -> 5 > [3.34482759 3.75862069 4.5862069  4.10344828 4.10344828 3. 5. 4.72413793 3. 5.]


In [87]:
# Create a results table
results_table = pd.DataFrame({
    'Outer Fold': np.arange(1, K_outer + 1),
    'Optimal Lambda': optimal_lambdas,
    'Optimal Hidden Units': optimal_hidden_units.astype(int),
    'Baseline Test Error': errors_outer['test']['baseline'].flatten(),
    'Regularized Test Error': errors_outer['test']['regularized'].flatten(),
    'ANN Test Error': errors_outer['test']['ann'].flatten()
})

# Display the table
print("\n" + "="*100)
print("TWO-LEVEL CROSS-VALIDATION RESULTS")
print("="*100)
print(results_table.to_string(index=False))

# Calculate mean
print("\n" + "="*100)
print("SUMMARY STATISTICS")
print("="*100)
summary_stats = pd.DataFrame({
    'Model': ['Baseline', 'Regularized', 'ANN'],
    'Mean Test Error': [
        errors_outer['test']['baseline'].mean(),
        errors_outer['test']['regularized'].mean(),
        errors_outer['test']['ann'].mean()
    ]
})
print(summary_stats.to_string(index=False))


TWO-LEVEL CROSS-VALIDATION RESULTS
 Outer Fold  Optimal Lambda  Optimal Hidden Units  Baseline Test Error  Regularized Test Error  ANN Test Error
          1        3.344828                    10             0.545593                0.317272        0.338392
          2        3.758621                     7             0.565115                0.262859        0.293071
          3        4.586207                     8             0.583031                0.269921        0.293636
          4        4.103448                     8             0.554766                0.244692        0.249750
          5        4.103448                    11             0.559984                0.257264        0.270077
          6        3.000000                     7             0.569823                0.236963        0.247756
          7        5.000000                    10             0.582204                0.250396        0.268466
          8        4.724138                     8             0.476979      

*Statistically evaluate if there is a significant performance difference between the fitted ANN, linear regression model and baseline using the methods described in chapter 11. These comparisons will be made pairwise (ANN vs. linear regression; ANN vs. baseline; linear regression vs. baseline*

We will perform the statistical evaluation following *setup I (11.3): Use the paired t-test described in 11.3.4*. Since we have the test errors for each fold stored in `errors_outer['test']`, we'll use these errors directly for our statistical comparisons.

In [88]:
# Extract test errors from each fold for all three models
ann_errors = errors_outer['test']['ann'].flatten()
ridge_errors = errors_outer['test']['regularized'].flatten()
baseline_errors = errors_outer['test']['baseline'].flatten()

In [89]:
def confidence_interval_comparison(errors_A, errors_B, alpha=0.05):
    z = errors_A - errors_B
    z_hat = np.mean(z)
    
    n = len(errors_A)
    nu = n - 1  # degrees of freedom
    
    sem = np.sqrt(sum(((z - z_hat)**2) / (n * nu)))
    CI = st.t.interval(1 - alpha, df=nu, loc=z_hat, scale=sem)
    
    t_stat = -np.abs(np.mean(z)) / st.sem(z)
    p_value = 2 * st.t.cdf(t_stat, df=nu)  # p-value
    
    return z_hat, CI, p_value

I defined the alpha.

In [90]:
alpha = 0.05

#### ANN vs linear regression

In [91]:
z_hat, CI, p_value = confidence_interval_comparison(ann_errors, ridge_errors, alpha=alpha)
print(f"Difference in loss between ANN and regularized linear regression: \nz_hat: {z_hat:.4f}, \nCI: [{CI[0]:.4f}, {CI[1]:.4f}], \np-value: {p_value}")

Difference in loss between ANN and regularized linear regression: 
z_hat: 0.0183, 
CI: [0.0129, 0.0236], 
p-value: 2.9771916546605427e-05


#### ANN vs baseline

In [92]:
z_hat, CI, p_value = confidence_interval_comparison(ann_errors, baseline_errors, alpha=alpha)
print(f"Difference in loss between ANN and baseline: \nz_hat: {z_hat:.4f}, \nCI: [{CI[0]:.4f}, {CI[1]:.4f}], \np-value: {p_value}")

Difference in loss between ANN and baseline: 
z_hat: -0.2797, 
CI: [-0.3095, -0.2499], 
p-value: 5.381953187585896e-09


#### Linear regression vs baseline

In [93]:
z_hat, CI, p_value = confidence_interval_comparison(ridge_errors, baseline_errors, alpha=alpha)
print(f"Difference in loss between regularized linear regression and baseline: \nz_hat: {z_hat:.4f}, \nCI: [{CI[0]:.4f}, {CI[1]:.4f}], \np-value: {p_value}")

Difference in loss between regularized linear regression and baseline: 
z_hat: -0.2980, 
CI: [-0.3273, -0.2687], 
p-value: 2.642311425218393e-09


LLM use: I have used an LLM to help me format the table for the TWO-LEVEL CROSS-VALIDATION RESULTS.